📅 **论文年份 (Year):2018 年**  
*GPipe: Easy Scaling with Micro-Batch Pipeline Parallelism — Huang et al. (NeurIPS 2019)*

# Paper 9: GPipe - Efficient Training of Giant Neural Networks using Pipeline Parallelism(论文 9:GPipe——利用流水线并行高效训练巨型神经网络)

**Paper**: Huang et al. (2019) - GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism

**Key Insight**: Training very large neural networks requires splitting them across multiple devices. GPipe introduces **pipeline parallelism** with **micro-batching** and **re-materialization** to efficiently train models that don't fit on a single accelerator.

**论文**:Huang 等人(2019)- GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism

**核心洞见**:训练超大规模神经网络需要将模型切分到多个设备上。GPipe 引入了**流水线并行(pipeline parallelism)**,并结合**微批次(micro-batching)**和**重计算(re-materialization)**,从而高效训练那些单个加速器装不下的模型。

## Core Concepts(核心概念)

### 1. Pipeline Parallelism(流水线并行)
- Split model into **K partitions** across K devices
- Each device holds consecutive layers
- Data flows through pipeline: Device 1 → Device 2 → ... → Device K

- 将模型切分为 **K 个分区(partition)**,分布到 K 个设备上
- 每个设备持有连续的若干层
- 数据依次流过流水线:设备 1 → 设备 2 → ... → 设备 K

### 2. Micro-Batching(微批次)
- Split mini-batch of size N into M micro-batches of size N/M
- Process micro-batches sequentially through pipeline
- **Reduces bubble time** (idle device time)

- 将大小为 N 的 mini-batch 切分为 M 个大小为 N/M 的微批次(micro-batch)
- 微批次依次通过流水线处理
- **减少气泡时间(bubble time)**(即设备空闲时间)

### 3. F-then-B Schedule(先前向后反向调度,F-then-B)
```
Forward all M micro-batches, then backward all M micro-batches
Device 1: F1 F2 F3 F4 ........... B4 B3 B2 B1
Device 2: .. F1 F2 F3 F4 ....... B4 B3 B2 B1
Device 3: .... F1 F2 F3 F4 ..... B4 B3 B2 B1
Device 4: ...... F1 F2 F3 F4 ... B4 B3 B2 B1
```

先对全部 M 个微批次做前向(forward),再对全部 M 个微批次做反向(backward)。

### 4. Re-materialization (Gradient Checkpointing)(重计算/梯度检查点)
- Don't store all activations (memory intensive)
- Only checkpoint partition boundaries
- Recompute intermediate activations during backward pass
- **Trade computation for memory**

- 不存储全部激活值(内存开销大)
- 只在分区边界处保存检查点(checkpoint)
- 在反向传播时重新计算中间激活值
- **用计算换内存**

### 5. Bubble Time(气泡时间)
- Fraction of time devices are idle: **(K-1) / (K-1 + M)**
- More micro-batches M → less bubble time
- More devices K → more bubble time

- 设备空闲时间占比:**(K-1) / (K-1 + M)**
- 微批次数 M 越多 → 气泡时间越少
- 设备数 K 越多 → 气泡时间越多

---

## Implementation Overview(实现概览)

We'll implement:
1. Model partitioning across "simulated" devices
2. Micro-batch splitting and scheduling
3. Forward and backward pass through pipeline
4. Gradient accumulation
5. Re-materialization for memory efficiency
6. Comparison with data parallelism
7. Bubble time analysis

Let's build it!

我们将实现:
1. 将模型切分到"模拟"设备上
2. 微批次切分与调度
3. 流水线中的前向与反向传播
4. 梯度累积(gradient accumulation)
5. 用于提升内存效率的重计算(re-materialization)
6. 与数据并行(data parallelism)的对比
7. 气泡时间分析

开始动手吧!

## 📖 论文导读

**🎯 这篇文章想解决什么问题(目的):** 深度学习有一条朴素的规律:模型越大,效果往往越好。可当模型大到一定程度,单块 GPU/TPU 的显存根本装不下。传统做法是把模型按层切开、分放到多块芯片上,但这样后面的芯片必须等前面的算完才能开工,大部分时间都在"干等",设备利用率非常低。GPipe 要解决的,就是"又要装得下大模型,又不让芯片闲着"这个难题。

**💡 主要贡献:** GPipe 提出了"微批次流水线并行":把一批训练数据切成许多小份(微批次),像工厂流水线一样依次送入各个设备,让所有芯片同时忙碌起来。同时配合"重计算"技巧——中间结果不保存、需要时再算一遍——用少量额外计算换来大幅的显存节省。最重要的是,这一切在数学上与不切分完全等价,不会影响训练效果。

**🔧 方法:** 先把网络按层切成 K 段,每段放到一块芯片上;再把每个训练批次切成 M 个微批次。前向传播时,微批次像接力棒一样在各段之间流动:第一个微批次进入第二段时,第一段就已开始处理第二个微批次。所有微批次跑完后,统一累积梯度、一次性更新参数。反向传播时只保留每段边界处的激活值,段内的中间结果现用现算,从而省下大量显存。

**🌟 意义:** GPipe 用实践证明了"把模型做大"这条路在工程上走得通——它训练出了当时创纪录的 5.57 亿参数图像模型和 60 亿参数多语言翻译模型,并展示了近乎线性的加速比。它是后来 GPT-3、PaLM 等巨型模型训练基础设施的思想源头之一:今天大模型训练标配的流水线并行(如 Megatron、DeepSpeed 中的实现)正是建立在这篇论文的基础之上。在这份阅读清单里,它代表着深度学习从"设计更好的模型"迈向"如何训练更大的模型"的关键一步。

## 🎯 核心结论 (Key Takeaways)

- **GPipe 的核心思想是"流水线 + 微批次"**:把大模型按层切成 K 段放到 K 台设备上,再把每个 mini-batch 切成 M 个微批次像流水线一样依次送入,让所有设备同时干活,而不是排队干等。
- **设备空转(气泡)有精确公式:(K-1)/(K-1+M)**。本 notebook 的实验表格验证了这一点:K=4 台设备时,M=8 个微批次气泡占比约 27.3%,把 M 加到 32 就降到 8.6%——微批次越多,流水线越"满"。
- **重计算(re-materialization)用"多算一遍"换"少存激活"**:只在分区边界存激活值,反向传播时再临时重算段内中间结果。notebook 中 M=8、K=4、每分区 3 层的例子里,激活内存从 960 MB 降到 320 MB,节省 3 倍(约等于每分区层数倍)。
- **数学上与不切分完全等价**:所有微批次的梯度累积取平均后一次性更新参数,训练效果不打折。notebook 的完整训练循环(12 层网络、K=4、M=8)跑了 3 个 epoch,损失逐轮下降,验证了整套机制端到端可用。
- **实用经验法则:M ≈ 4×K**。notebook 最后的配置对照表显示,按此法则(如 K=4/M=16、K=8/M=32)流水线效率稳定在约 88.9%;论文正是靠这套方法训练出了当时创纪录的 5.57 亿参数 AmoebaNet(ImageNet top-1 84.4%)和 60 亿参数翻译模型。
- **一句话带走**:GPipe 证明了"模型装不下就切开、切开还能不闲着"在工程上完全可行,它是今天 GPT 级大模型训练标配的流水线并行(Megatron、DeepSpeed 等)的思想源头。


## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为"算过的东西再算一遍"是纯浪费**,但 GPipe 的重计算(re-materialization)恰恰反其道而行:前向传播时故意扔掉段内的中间激活值,反向传播需要时再重算一遍。本 notebook 的实验(K=4、M=8、每分区 3 层)显示,激活内存从 960 MB 直降到 320 MB——用约 1/3 的额外计算换来 3 倍内存节省,在"显存是硬上限、算力还有富余"的加速器上,这笔买卖稳赚不赔。
- **常识认为把一个 batch 切碎、分成许多小份轮流算,训练结果肯定会变**,但 GPipe 把 mini-batch 切成 M 个微批次流水执行后,所有梯度先累积、取平均、再一次性更新参数——数学上与不切分完全等价。notebook 里 12 层网络、K=4、M=8 的完整训练循环损失照常逐轮下降,证明"切碎"不掉一分精度。
- **常识认为设备加得越多训练越快**,但流水线并行里恰恰相反:气泡(空转)占比是 (K-1)/(K-1+M),K 越大气泡反而越大——盲目加机器只会让大家一起排队干等。真正的解药是多切微批次:notebook 的实验表格显示,K=4 时把 M 从 8 加到 32,气泡占比从 27.3% 降到 8.6%。也就是说,提高效率靠的不是"堆硬件",而是"把活切得更碎"。
- **常识认为"流水线"听起来像复杂的系统工程,肯定要为特定模型定制**,但 GPipe 的方案惊人地通用:只要网络能按层切开,任何架构都能直接套用,不需要改动模型本身或训练算法。一个简单的经验法则 M ≈ 4×K(notebook 配置对照表中效率稳定在约 88.9%)就足以让它开箱即用地训练出当时创纪录的 60 亿参数模型。


#### 💻 代码解读

**做什么:** 导入本笔记本需要的所有工具库,并固定随机种子,保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`(矩阵计算)和 `matplotlib.pyplot`(画图),以及 `dataclass`、类型标注等辅助工具;
- 用 `np.random.seed(42)` 固定随机数种子,就像"每次洗牌都按同一套顺序",方便复现实验;
- 最后打印 NumPy 版本,确认环境就绪。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Callable
from dataclasses import dataclass
import time
from collections import defaultdict

np.random.seed(42)

print("Libraries imported successfully!")
print("NumPy version:", np.__version__)

# Section 1: Model Partitioning and Pipeline Structure(第 1 节:模型切分与流水线结构)

The first step in GPipe is to partition a large model into K segments, each assigned to a different device.

GPipe 的第一步是把一个大模型切分成 K 段,每段分配给一个不同的设备。

## Partitioning Strategy(切分策略)

For a model with L layers:
- **Uniform partitioning**: Each partition gets ~L/K layers
- **Balanced partitioning**: Partition by computation time or memory

We'll implement a simple multi-layer network and partition it uniformly.

对于一个有 L 层的模型:
- **均匀切分(uniform partitioning)**:每个分区约获得 L/K 层
- **均衡切分(balanced partitioning)**:按计算时间或内存进行切分

我们将实现一个简单的多层网络,并对其进行均匀切分。

#### 💻 代码解读

**做什么:** 搭建 GPipe 的地基:定义"层"和"分区"两个积木,并把一个 12 层的神经网络切成 4 份,模拟分给 4 台设备。

**怎么做:**
- `Layer` 类表示一层网络:`forward` 做线性变换加激活(ReLU/tanh/线性),`backward` 反向算出权重梯度 `dW`、偏置梯度 `db` 和传给上一层的 `dx`;
- `Partition` 类是"一台设备负责的一段层",它的 `forward`/`backward` 依次穿过自己名下的所有层,并保存中间激活值供反向传播用;
- `create_model` 按给定维度随机初始化各层权重(He 初始化),`partition_model` 把层平均切成 K 份,就像把一条流水线分给 K 个车间;
- 最后实际创建 12 层网络并切成 K=4 个分区,打印每台"设备"分到几层。

In [ ]:
@dataclass
class Layer:
    """A single neural network layer."""
    W: np.ndarray  # Weight matrix
    b: np.ndarray  # Bias vector
    activation: str = 'relu'  # 'relu', 'tanh', or 'linear'
    
    def forward(self, x: np.ndarray, store_activation: bool = True) -> Tuple[np.ndarray, np.ndarray]:
        """Forward pass: z = W @ x + b, a = activation(z)"""
        # @是矩阵乘法,形状:(batch, in_dim) @ (in_dim, out_dim) -> (batch, out_dim);+b靠广播加到每一行
        z = x @ self.W + self.b  # Linear transformation
        
        # Apply activation function
        if self.activation == 'relu':
            a = np.maximum(0, z)
        elif self.activation == 'tanh':
            a = np.tanh(z)
        elif self.activation == 'linear':
            a = z
        else:
            raise ValueError(f"Unknown activation: {self.activation}")
        
        return a, z if store_activation else None
    
    def backward(self, da: np.ndarray, z: np.ndarray, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Backward pass: compute gradients."""
        # Activation gradient
        # 链式法则第一步:dz = da * σ'(z);ReLU的导数是指示函数(z>0),布尔数组直接当0/1用
        if self.activation == 'relu':
            dz = da * (z > 0)
        elif self.activation == 'tanh':
            dz = da * (1 - np.tanh(z)**2)
        elif self.activation == 'linear':
            dz = da
        else:
            raise ValueError(f"Unknown activation: {self.activation}")
        
        # Parameter gradients
        # dW = x^T @ dz,形状:(in_dim, batch) @ (batch, out_dim) -> (in_dim, out_dim),自动对batch求和
        dW = x.T @ dz
        # 偏置梯度沿batch维(axis=0)求和,因为b被广播到了每个样本
        db = np.sum(dz, axis=0)
        
        # Input gradient (for previous layer)
        # dx传给前一层继续反向传播,形状回到(batch, in_dim)
        dx = dz @ self.W.T
        
        return dx, dW, db


# Partition是GPipe的核心概念:把连续的若干层打包分配给一个"设备"(这里用单机模拟)
@dataclass
class Partition:
    """A partition of the model (subset of layers assigned to one device)."""
    device_id: int
    layers: List[Layer]
    
    def forward(self, x: np.ndarray, store_activations: bool = True) -> Tuple[np.ndarray, List[Tuple]]:
        """Forward pass through all layers in this partition."""
        activations = []  # Store (x, z) for each layer if needed
        
        current = x
        for layer in self.layers:
            if store_activations:
                activations.append(current)  # Store input to this layer
            
            current, z = layer.forward(current, store_activation=store_activations)
            
            if store_activations:
                activations.append(z)  # Store pre-activation
        
        return current, activations
    
    def backward(self, dout: np.ndarray, activations: List) -> Tuple[np.ndarray, List[Tuple]]:
        """Backward pass through all layers in this partition."""
        gradients = []  # Store (dW, db) for each layer
        
        da = dout
        # Go through layers in reverse
        # range(n-1, -1, -1)从最后一层倒序遍历到第0层,反向传播必须逆着前向顺序
        for i in range(len(self.layers) - 1, -1, -1):
            layer = self.layers[i]
            
            # Get stored activations
            # 前向时每层存了两项(输入x和预激活z),所以第i层的数据在2i和2i+1位置
            x = activations[2*i]      # Input to this layer
            z = activations[2*i + 1]  # Pre-activation
            
            # Compute gradients
            da, dW, db = layer.backward(da, z, x)
            # insert(0,...)把梯度插到列表头,保证最终顺序与层顺序一致
            gradients.insert(0, (dW, db))
        
        return da, gradients  # da is gradient w.r.t. partition input


def create_model(layer_dims: List[int], activations: List[str]) -> List[Layer]:
    """Create a multi-layer neural network.
    
    Args:
        layer_dims: [input_dim, hidden1, hidden2, ..., output_dim]
        activations: Activation for each layer
    """
    layers = []
    for i in range(len(layer_dims) - 1):
        # He初始化:方差2/fan_in,配合ReLU防止深层网络信号逐层衰减或爆炸
        W = np.random.randn(layer_dims[i], layer_dims[i+1]) * np.sqrt(2.0 / layer_dims[i])
        b = np.zeros(layer_dims[i+1])
        layers.append(Layer(W, b, activations[i]))
    return layers


# 对应论文中的模型切分:把L层均匀切成K段,每段放到一个设备上(模型并行的基础)
def partition_model(layers: List[Layer], num_partitions: int) -> List[Partition]:
    """Partition layers uniformly across devices."""
    num_layers = len(layers)
    layers_per_partition = num_layers // num_partitions
    
    partitions = []
    for k in range(num_partitions):
        start = k * layers_per_partition
        if k == num_partitions - 1:
            # Last partition gets any remaining layers
            end = num_layers
        else:
            end = (k + 1) * layers_per_partition
        
        partition_layers = layers[start:end]
        partitions.append(Partition(device_id=k, layers=partition_layers))
    
    return partitions


# Example: Create and partition a 12-layer network
# 列表拼接技巧:[256]*10生成10个256,+把三段列表连成完整的维度序列
layer_dims = [128] + [256] * 10 + [10]  # Input=128, 10 hidden layers of 256, output=10
activations = ['relu'] * 10 + ['linear']  # ReLU for hidden, linear for output

model_layers = create_model(layer_dims, activations)
print(f"Created model with {len(model_layers)} layers")

# Partition across 4 "devices"
K = 4
partitions = partition_model(model_layers, K)

print(f"\nPartitioned model into {K} partitions:")
for i, partition in enumerate(partitions):
    print(f"  Device {i}: {len(partition.layers)} layers")

print("\n✓ Model partitioning complete!")

# Section 2: Micro-Batching Strategy(第 2 节:微批次策略)

GPipe splits each mini-batch into M **micro-batches** to improve pipeline utilization.

GPipe 将每个 mini-batch 切分为 M 个**微批次(micro-batch)**,以提高流水线利用率。

## Why Micro-Batching?(为什么要用微批次?)

Without micro-batching:
```
Device 1: [Forward] .................... [Backward]
Device 2:          [Forward] .......... [Backward]
Device 3:                   [Forward] [Backward]
          ^^^^^^^^                     ^^^^^^^^^^
          Bubble                       Bubble
```

With M micro-batches:
```
Device 1: F1 F2 F3 F4 ........... B4 B3 B2 B1
Device 2:    F1 F2 F3 F4 ....... B4 B3 B2 B1
Device 3:       F1 F2 F3 F4 .... B4 B3 B2 B1
          ^^                              ^^
          Smaller bubble
```

**Bubble fraction**: (K-1) / (K-1 + M)
- More micro-batches → less bubble time
- But more micro-batches → more overhead

不使用微批次时(上方第一张图):流水线中存在大段气泡(bubble,设备空闲)。使用 M 个微批次后(第二张图):气泡明显变小。

**气泡占比(bubble fraction)**:(K-1) / (K-1 + M)
- 微批次越多 → 气泡时间越少
- 但微批次越多 → 额外开销也越大

#### 💻 代码解读

**做什么:** 实现微批次切分,并用公式量化"流水线气泡"(设备空转)的占比,展示微批次越多、空转越少。

**怎么做:**
- `split_into_microbatches` 把一个 mini-batch 平均切成 M 个小份(微批次),要求批大小能被 M 整除,就像把一大箱货拆成 M 个小包裹依次发货;
- `compute_bubble_fraction` 按论文公式 `(K-1)/(K-1+M)` 计算气泡占比:K 是设备数,M 是微批次数;
- 打印一张 K × M 的对照表:比如 K=4、M=8 时设备约 27% 时间在空转,M=32 时降到 8.6%;
- 最后实际把 32 个样本切成 8 个微批次,打印每个微批次的形状验证切分正确。

In [ ]:
# GPipe关键思想一:把mini-batch再切成M个micro-batch,让多个设备能同时处理不同的小块
def split_into_microbatches(X: np.ndarray, y: np.ndarray, num_microbatches: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Split mini-batch into micro-batches.
    
    Args:
        X: Input data (batch_size, features)
        y: Labels (batch_size, ...)
        num_microbatches: M (number of micro-batches)
    
    Returns:
        List of (X_micro, y_micro) tuples
    """
    batch_size = X.shape[0]
    microbatch_size = batch_size // num_microbatches
    
    if batch_size % num_microbatches != 0:
        raise ValueError(f"Batch size {batch_size} must be divisible by num_microbatches {num_microbatches}")
    
    microbatches = []
    for m in range(num_microbatches):
        start = m * microbatch_size
        end = (m + 1) * microbatch_size
        # 切片X[start:end]沿batch维取第m个小块,不复制特征维
        microbatches.append((X[start:end], y[start:end]))
    
    return microbatches


def compute_bubble_fraction(K: int, M: int) -> float:
    """Theoretical bubble fraction for GPipe.
    
    Formula: (K - 1) / (K - 1 + M)
    
    Args:
        K: Number of devices/partitions
        M: Number of micro-batches
    """
    # 气泡=流水线启动/排空阶段的空闲:填满K级流水线需要K-1步,之后M个micro-batch连续流过
    # 所以M越大,空闲占比越小;这是论文中著名的bubble公式
    return (K - 1) / (K - 1 + M)


# Example: Analyze bubble fraction
K_values = [2, 4, 8, 16]
M_values = [1, 2, 4, 8, 16, 32, 64]

print("Bubble Fraction Analysis:")
print("\nM (micro-batches) →")
print("K ↓\t" + "\t".join(f"{M:d}" for M in M_values))
print("-" * 80)

for K in K_values:
    row = f"{K}\t"
    for M in M_values:
        bubble = compute_bubble_fraction(K, M)
        row += f"{bubble:.3f}\t"
    print(row)

print("\nKey observations:")
print("  - More devices (K) → more bubble time (devices wait for pipeline)")
print("  - More micro-batches (M) → less bubble time (pipeline stays full)")
print("  - With K=4, M=8: bubble fraction = 27.3% (device idle 27% of time)")
print("  - With K=4, M=32: bubble fraction = 8.6% (much better!)")

# Example micro-batching
batch_size = 32
M = 8
X_batch = np.random.randn(batch_size, 128)
y_batch = np.random.randint(0, 10, batch_size)

microbatches = split_into_microbatches(X_batch, y_batch, M)
print(f"\n\nSplit batch of {batch_size} into {M} micro-batches:")
for i, (X_m, y_m) in enumerate(microbatches):
    print(f"  Micro-batch {i}: X shape {X_m.shape}, y shape {y_m.shape}")

print("\n✓ Micro-batching complete!")

# Section 3: Forward Pass Through Pipeline (F-then-B Schedule)(第 3 节:流水线前向传播(F-then-B 调度))

GPipe uses an **F-then-B schedule**:
1. Forward all M micro-batches through pipeline
2. Backward all M micro-batches through pipeline (in reverse order)

GPipe 采用**先前向后反向(F-then-B)调度**:
1. 先让全部 M 个微批次通过流水线完成前向传播
2. 再让全部 M 个微批次(按相反顺序)通过流水线完成反向传播

## Timeline Example (K=3 devices, M=4 micro-batches):(时间线示例(K=3 个设备,M=4 个微批次))

```
Time →  0   1   2   3   4   5   6   7   8   9   10  11  12
Dev 0:  F0  F1  F2  F3  ... ... ... B3  B2  B1  B0
Dev 1:  ... F0  F1  F2  F3  ... ... ... B3  B2  B1  B0
Dev 2:  ... ... F0  F1  F2  F3  ... ... ... B3  B2  B1  B0
```

Key:
- **F0** = Forward micro-batch 0
- **B3** = Backward micro-batch 3
- **...** = Bubble (device idle)

图例说明:
- **F0** = 微批次 0 的前向传播
- **B3** = 微批次 3 的反向传播
- **...** = 气泡(设备空闲)

#### 💻 代码解读

**做什么:** 实现 GPipe 的核心调度器:按"先全部前向、再全部反向"(F-then-B)的顺序,让 M 个微批次依次流过 K 台设备,并记录执行时间线。

**怎么做:**
- `PipelineEvent` 记录"哪个时刻、哪台设备、对哪个微批次做了前向还是反向",相当于流水线上的打卡记录;
- `GPipePipeline.forward_pipeline` 让每个微批次依次穿过所有分区,保存各分区的激活值,并给每一步打上时间戳;
- `backward_pipeline` 以均方误差(MSE)算出输出端的梯度,再按微批次倒序、分区倒序往回传,收集每层梯度;
- `get_timeline_matrix` 把打卡记录变成 K×T 矩阵(正数=前向,负数=反向,0=空转气泡),供后面画图;
- 最后用 4 个微批次实测一遍前向+反向,打印事件数和总时间步数。

In [ ]:
@dataclass
class PipelineEvent:
    """Records when a device executes an operation."""
    time_step: int
    device_id: int
    operation: str  # 'forward' or 'backward'
    microbatch_id: int


# GPipe采用F-then-B调度:先做完所有micro-batch的前向,再统一做反向(区别于1F1B等交错调度)
class GPipePipeline:
    """GPipe pipeline with F-then-B schedule."""
    
    def __init__(self, partitions: List[Partition]):
        self.partitions = partitions
        self.K = len(partitions)  # Number of devices
        
        # For tracking execution timeline
        self.events = []  # List of PipelineEvent
    
    def forward_pipeline(self, microbatches: List[Tuple[np.ndarray, np.ndarray]], 
                        store_activations: bool = True) -> Tuple[List[np.ndarray], List[List]]:
        """Forward pass: process all micro-batches through pipeline.
        
        Returns:
            outputs: List of final outputs for each micro-batch
            all_activations: List of activation lists (one per micro-batch)
        """
        M = len(microbatches)
        
        # Storage for outputs and activations
        outputs = [None] * M
        # 列表推导式构造M×K的二维列表;不能写[[None]*K]*M,那样每行会共享同一个列表对象
        all_activations = [[None] * self.K for _ in range(M)]  # [microbatch][partition]
        
        # F-then-B schedule: Forward all micro-batches
        time_step = 0
        
        # 注意:这是串行模拟——真实GPipe中不同设备会并行处理不同micro-batch,这里只记录事件用于分析
        for m in range(M):
            X_micro, y_micro = microbatches[m]
            current = X_micro
            
            # Forward through each partition
            for k, partition in enumerate(self.partitions):
                self.events.append(PipelineEvent(time_step, k, 'forward', m))
                
                # current在分区间传递,相当于真实系统中设备k把激活发送给设备k+1
                current, activations = partition.forward(current, store_activations)
                all_activations[m][k] = activations
                
                time_step += 1
            
            outputs[m] = current
        
        return outputs, all_activations
    
    def backward_pipeline(self, outputs: List[np.ndarray], 
                         labels: List[np.ndarray],
                         all_activations: List[List]) -> List[List[List[Tuple]]]:
        """Backward pass: process all micro-batches in reverse.
        
        Returns:
            all_gradients: [microbatch][partition][(dW, db) for each layer]
        """
        M = len(outputs)
        
        # Storage for gradients
        all_gradients = [[None] * self.K for _ in range(M)]
        
        # Find current time step (after forward passes)
        time_step = max(e.time_step for e in self.events) + 1
        
        # Backward all micro-batches in reverse order
        for m in range(M - 1, -1, -1):
            # Compute loss gradient (simple MSE for demonstration)
            # MSE损失L=mean((y_hat-y)^2)对输出的导数:2(y_hat-y)/N,这是反向传播的起点
            dout = 2 * (outputs[m] - labels[m]) / labels[m].shape[0]
            
            # Backward through each partition in reverse
            # 梯度从最后一个分区(设备K-1)流回第一个分区,与前向方向相反
            for k in range(self.K - 1, -1, -1):
                partition = self.partitions[k]
                activations = all_activations[m][k]
                
                self.events.append(PipelineEvent(time_step, k, 'backward', m))
                
                dout, gradients = partition.backward(dout, activations)
                all_gradients[m][k] = gradients
                
                time_step += 1
        
        return all_gradients
    
    def get_timeline_matrix(self) -> np.ndarray:
        """Convert events to a K×T matrix for visualization.
        
        Matrix values:
            0 = bubble (idle)
            m+1 = forward micro-batch m
            -(m+1) = backward micro-batch m
        """
        max_time = max(e.time_step for e in self.events) + 1
        timeline = np.zeros((self.K, max_time))
        
        # 编码技巧:正数表示前向、负数表示反向、0表示空闲气泡,方便后面用颜色画时间线
        for event in self.events:
            value = event.microbatch_id + 1
            if event.operation == 'backward':
                value = -value
            timeline[event.device_id, event.time_step] = value
        
        return timeline


# Test forward pass
print("Testing GPipe forward pass...\n")

# Create pipeline
pipeline = GPipePipeline(partitions)

# Create micro-batches
M = 4
batch_size = 16
X_batch = np.random.randn(batch_size, 128)
# one-hot技巧:用整数标签对单位矩阵做花式索引,直接取出对应的one-hot行,形状(batch, 10)
y_batch_onehot = np.eye(10)[np.random.randint(0, 10, batch_size)]

microbatches = split_into_microbatches(X_batch, y_batch_onehot, M)

# Forward pass
outputs, all_activations = pipeline.forward_pipeline(microbatches)

print(f"Processed {M} micro-batches through {pipeline.K} devices")
print(f"Output shapes: {[out.shape for out in outputs]}")
print(f"Total forward events: {len([e for e in pipeline.events if e.operation == 'forward'])}")

# Backward pass
labels = [mb[1] for mb in microbatches]
all_gradients = pipeline.backward_pipeline(outputs, labels, all_activations)

print(f"Total backward events: {len([e for e in pipeline.events if e.operation == 'backward'])}")
print(f"\nTotal time steps: {max(e.time_step for e in pipeline.events) + 1}")

print("\n✓ Pipeline forward and backward passes complete!")

# Section 4: Gradient Accumulation Across Micro-Batches(第 4 节:跨微批次的梯度累积)

After processing all M micro-batches, we need to:
1. **Accumulate gradients** from all micro-batches
2. **Average** them (since they're from the same mini-batch)
3. **Apply** the accumulated gradient to update parameters

This is equivalent to processing the entire mini-batch at once, but with better pipeline utilization!

在处理完全部 M 个微批次之后,我们需要:
1. **累积**所有微批次的梯度
2. 对它们**取平均**(因为它们来自同一个 mini-batch)
3. **应用**累积后的梯度来更新参数

这与一次性处理整个 mini-batch 等价,但流水线利用率更高!

#### 💻 代码解读

**做什么:** 把 M 个微批次各自算出的梯度加起来取平均,再用平均梯度一次性更新模型参数——这保证 GPipe 与单机训练在数学上完全等价。

**怎么做:**
- `accumulate_gradients` 对每个分区的每一层,把 M 个微批次的 `dW`、`db` 求和再除以 M,相当于"把大家的意见汇总后取平均";
- `apply_gradients` 用最基本的 SGD 规则 `W -= learning_rate * dW` 更新每层的权重和偏置;
- 用上一格算出的 `all_gradients` 实测:打印各分区梯度的形状和范数,再对比更新前后第一层权重的变化量,确认参数确实被更新了。

In [ ]:
# GPipe关键性质:各micro-batch的梯度累加后再更新,数学上等价于直接用整个mini-batch训练
def accumulate_gradients(all_gradients: List[List[List[Tuple]]]) -> List[List[Tuple]]:
    """Accumulate and average gradients from all micro-batches.
    
    Args:
        all_gradients: [microbatch][partition][(dW, db) per layer]
    
    Returns:
        accumulated: [partition][(dW, db) per layer] - averaged over micro-batches
    """
    M = len(all_gradients)  # Number of micro-batches
    K = len(all_gradients[0])  # Number of partitions
    
    # Initialize accumulated gradients (copy structure from first micro-batch)
    accumulated = []
    for k in range(K):
        partition_grads = []
        for layer_idx in range(len(all_gradients[0][k])):
            # Sum gradients across micro-batches
            # sum接收生成器表达式,把M个梯度数组逐元素相加(NumPy数组支持+)
            dW_sum = sum(all_gradients[m][k][layer_idx][0] for m in range(M))
            db_sum = sum(all_gradients[m][k][layer_idx][1] for m in range(M))
            
            # Average (since micro-batches are part of same mini-batch)
            # 除以M取平均,使梯度尺度与不切分micro-batch时一致,学习率无需调整
            dW_avg = dW_sum / M
            db_avg = db_sum / M
            
            partition_grads.append((dW_avg, db_avg))
        
        accumulated.append(partition_grads)
    
    return accumulated


def apply_gradients(partitions: List[Partition], gradients: List[List[Tuple]], learning_rate: float):
    """Apply accumulated gradients to update parameters.
    
    Args:
        partitions: List of model partitions
        gradients: [partition][(dW, db) per layer]
        learning_rate: Learning rate for SGD
    """
    for k, partition in enumerate(partitions):
        partition_grads = gradients[k]
        
        for layer_idx, layer in enumerate(partition.layers):
            dW, db = partition_grads[layer_idx]
            
            # SGD update
            # 同步更新:所有micro-batch反向完成后才更新一次参数(GPipe不存在梯度陈旧问题)
            layer.W -= learning_rate * dW
            layer.b -= learning_rate * db


# Test gradient accumulation
print("Testing gradient accumulation...\n")

# We already have all_gradients from previous cell
accumulated_grads = accumulate_gradients(all_gradients)

print(f"Accumulated gradients for {len(accumulated_grads)} partitions:")
for k, partition_grads in enumerate(accumulated_grads):
    print(f"  Partition {k}: {len(partition_grads)} layers")
    for i, (dW, db) in enumerate(partition_grads[:2]):  # Show first 2 layers
        print(f"    Layer {i}: dW shape {dW.shape}, db shape {db.shape}")
        print(f"             dW norm: {np.linalg.norm(dW):.6f}, db norm: {np.linalg.norm(db):.6f}")

# Apply gradients
learning_rate = 0.01
old_W = partitions[0].layers[0].W.copy()

apply_gradients(partitions, accumulated_grads, learning_rate)

new_W = partitions[0].layers[0].W
weight_change = np.linalg.norm(new_W - old_W)

print(f"\nApplied gradients with learning rate {learning_rate}")
print(f"Weight change (first layer): {weight_change:.6f}")

print("\n✓ Gradient accumulation and application complete!")

# Section 5: Re-materialization (Gradient Checkpointing)(第 5 节:重计算(梯度检查点))

**Problem**: Storing activations for all M micro-batches across K partitions requires O(M × K × layer_memory) memory.

**Solution**: **Re-materialization** (gradient checkpointing)
- Only checkpoint activations at **partition boundaries**
- During backward pass, **recompute** intermediate activations
- Trade: ~33% extra computation for ~K× less memory

**问题**:为 K 个分区上的全部 M 个微批次存储激活值,需要 O(M × K × layer_memory) 的内存。

**解决方案**:**重计算(re-materialization,即梯度检查点 gradient checkpointing)**
- 只在**分区边界**处保存激活值检查点
- 在反向传播时**重新计算**中间激活值
- 代价:约 33% 的额外计算,换来约 K 倍的内存节省

## Memory Comparison(内存对比)

**Without re-materialization**:
- Store activations for all layers in all partitions
- Memory: O(M × L) where L = total layers

**With re-materialization**:
- Store activations only at partition boundaries
- Memory: O(M × K) where K = number of partitions (K << L)
- Recompute intermediate activations as needed

**不使用重计算时**:
- 存储所有分区中所有层的激活值
- 内存:O(M × L),其中 L 为总层数

**使用重计算时**:
- 只存储分区边界处的激活值
- 内存:O(M × K),其中 K 为分区数(K << L)
- 按需重新计算中间激活值

#### 💻 代码解读

**做什么:** 实现重计算(re-materialization,即梯度检查点):前向时只保存每个分区的输入,反向时再临时重算中间激活,用"多算一遍"换"省大量内存"。

**怎么做:**
- `GPipePipelineWithRemat.forward_pipeline_remat` 前向时设 `store_activations=False`,只把每个分区的输入存进 `boundary_inputs`,就像旅途中只在关卡处拍照留档,不记录沿途每一步;
- `backward_pipeline_remat` 反向传到某个分区时,先用存好的边界输入重新跑一遍该分区的前向、恢复中间激活,再用它们计算梯度;
- `estimate_memory_usage` 估算两种模式的内存:不重计算要存"所有层"的激活,重计算只存"K 个边界",内存差约等于每分区的层数倍;
- 实测 M=8、K=4、每分区 3 层的例子:内存从 960 MB 降到 320 MB,省 3 倍。

In [ ]:
# GPipe关键思想二:重计算(re-materialization)。前向只存每个分区的输入,
# 反向时重新算一遍分区内部激活——用约1/3的额外计算换取大幅省显存
class GPipePipelineWithRemat:
    """GPipe with re-materialization (gradient checkpointing)."""
    
    def __init__(self, partitions: List[Partition]):
        self.partitions = partitions
        self.K = len(partitions)
        self.events = []
    
    def forward_pipeline_remat(self, microbatches: List[Tuple[np.ndarray, np.ndarray]]) -> Tuple[List, List]:
        """Forward pass with re-materialization: only store partition boundary activations.
        
        Returns:
            outputs: Final outputs for each micro-batch
            boundary_inputs: Inputs to each partition (for recomputation)
        """
        M = len(microbatches)
        
        outputs = [None] * M
        # Only store inputs to each partition (boundary activations)
        boundary_inputs = [[None] * self.K for _ in range(M)]
        
        time_step = 0
        
        for m in range(M):
            X_micro, y_micro = microbatches[m]
            current = X_micro
            
            for k, partition in enumerate(self.partitions):
                # Store input to this partition (boundary)
                # 只保存分区边界激活(checkpoint),.copy()防止后续计算原地修改它
                boundary_inputs[m][k] = current.copy()
                
                self.events.append(PipelineEvent(time_step, k, 'forward', m))
                
                # Forward pass WITHOUT storing intermediate activations
                current, _ = partition.forward(current, store_activations=False)
                
                time_step += 1
            
            outputs[m] = current
        
        return outputs, boundary_inputs
    
    def backward_pipeline_remat(self, outputs: List[np.ndarray],
                                labels: List[np.ndarray],
                                boundary_inputs: List[List]) -> List[List[List[Tuple]]]:
        """Backward pass with re-materialization: recompute activations as needed."""
        M = len(outputs)
        all_gradients = [[None] * self.K for _ in range(M)]
        
        time_step = max(e.time_step for e in self.events) + 1
        
        for m in range(M - 1, -1, -1):
            dout = 2 * (outputs[m] - labels[m]) / labels[m].shape[0]
            
            for k in range(self.K - 1, -1, -1):
                partition = self.partitions[k]
                
                self.events.append(PipelineEvent(time_step, k, 'backward', m))
                
                # RECOMPUTE activations for this partition
                # 从保存的边界输入出发,重新前向一次,恢复反向所需的全部中间激活
                partition_input = boundary_inputs[m][k]
                _, activations = partition.forward(partition_input, store_activations=True)
                
                # Now compute gradients using recomputed activations
                dout, gradients = partition.backward(dout, activations)
                all_gradients[m][k] = gradients
                
                time_step += 1
        
        return all_gradients


def estimate_memory_usage(M: int, K: int, layers_per_partition: int, 
                         activation_size_mb: float, with_remat: bool) -> float:
    """Estimate memory usage with and without re-materialization.
    
    Args:
        M: Number of micro-batches
        K: Number of partitions
        layers_per_partition: Average layers per partition
        activation_size_mb: Memory for one layer's activations (MB)
        with_remat: Use re-materialization?
    
    Returns:
        Estimated memory in MB
    """
    if with_remat:
        # Only store boundary inputs (K per micro-batch)
        # 显存从O(M×总层数)降到O(M×K):每个micro-batch每个分区只存1份边界激活
        return M * K * activation_size_mb
    else:
        # Store all intermediate activations
        total_layers = K * layers_per_partition
        return M * total_layers * activation_size_mb


# Test re-materialization
print("Testing re-materialization...\n")

# Create fresh pipeline with remat
pipeline_remat = GPipePipelineWithRemat(partitions)

# Forward with remat
outputs_remat, boundary_inputs = pipeline_remat.forward_pipeline_remat(microbatches)

print("Forward pass with re-materialization:")
print(f"  Stored boundary inputs: {len(boundary_inputs)} micro-batches × {len(boundary_inputs[0])} partitions")
print(f"  Boundary input shapes: {[bi[0].shape for bi in boundary_inputs]}")

# Backward with remat
gradients_remat = pipeline_remat.backward_pipeline_remat(outputs_remat, labels, boundary_inputs)

print(f"\nBackward pass with re-materialization:")
print(f"  Gradients computed: {len(gradients_remat)} micro-batches × {len(gradients_remat[0])} partitions")

# Memory analysis
print("\n" + "="*70)
print("Memory Usage Comparison")
print("="*70)

M_test = 8
K_test = 4
layers_per_partition = 3
activation_size_mb = 10  # MB per layer activation

mem_without = estimate_memory_usage(M_test, K_test, layers_per_partition, activation_size_mb, with_remat=False)
mem_with = estimate_memory_usage(M_test, K_test, layers_per_partition, activation_size_mb, with_remat=True)

print(f"\nConfiguration: M={M_test}, K={K_test}, {layers_per_partition} layers/partition")
print(f"  Without re-materialization: {mem_without:.1f} MB")
print(f"  With re-materialization:    {mem_with:.1f} MB")
print(f"  Memory savings:             {mem_without / mem_with:.1f}×")

print("\n✓ Re-materialization complete!")

# Section 6: Pipeline Schedule Visualization and Bubble Analysis(第 6 节:流水线调度可视化与气泡分析)

Let's visualize the F-then-B schedule and quantify bubble time.

下面我们对 F-then-B 调度进行可视化,并对气泡时间进行量化分析。

#### 💻 代码解读

**做什么:** 把流水线的执行时间表画成甘特图,直观看到前向、反向和"气泡"(空白格),并对比实际气泡率与理论公式。

**怎么做:**
- `visualize_pipeline_schedule` 取出 K×T 时间线矩阵,给每个格子上色:红色系=前向(标 F0、F1……),蓝色系=反向(标 B0、B1……),白色=设备空转的气泡;
- 每个格子用 `plt.Rectangle` 画方块并标注微批次编号,纵轴是设备、横轴是时间步,右上角配图例;
- `compute_actual_bubble_time` 统计矩阵中 0(空转格)占总格子的比例,得到实际气泡率;
- 最后画出 K=4、M=4 配置的调度图,并把实际气泡率与理论值 `(K-1)/(K-1+M)` 放在一起对比,算出流水线效率。

In [ ]:
def visualize_pipeline_schedule(pipeline: GPipePipeline, title: str = "GPipe Schedule (F-then-B)"):
    """Visualize pipeline execution timeline."""
    timeline = pipeline.get_timeline_matrix()
    K, T = timeline.shape
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Create color map
    # Positive = forward (warm colors), negative = backward (cool colors), 0 = bubble (white)
    # 时间线里的最大绝对值就是micro-batch数M(编码为±(m+1))
    M = int(np.max(np.abs(timeline)))
    # 从色图中均匀取M种深浅:每个micro-batch一种颜色,前向用红色系、反向用蓝色系
    colors_forward = plt.cm.Reds(np.linspace(0.3, 0.9, M))
    colors_backward = plt.cm.Blues(np.linspace(0.3, 0.9, M))
    
    # Plot timeline
    # 逐格绘制K×T的甘特图:行=设备,列=时间步,颜色区分前向/反向/气泡
    for k in range(K):
        for t in range(T):
            val = timeline[k, t]
            if val > 0:  # Forward
                color = colors_forward[int(val) - 1]
                label = f'F{int(val)-1}'
            elif val < 0:  # Backward
                color = colors_backward[int(-val) - 1]
                label = f'B{int(-val)-1}'
            else:  # Bubble
                color = 'white'
                label = ''
            
            rect = plt.Rectangle((t, k), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            
            if label:
                ax.text(t + 0.5, k + 0.5, label, ha='center', va='center', 
                       fontsize=9, fontweight='bold')
    
    ax.set_xlim(0, T)
    ax.set_ylim(0, K)
    ax.set_xlabel('Time Step', fontsize=12)
    ax.set_ylabel('Device', fontsize=12)
    ax.set_yticks(np.arange(K) + 0.5)
    ax.set_yticklabels([f'Device {k}' for k in range(K)])
    ax.set_xticks(np.arange(T) + 0.5)
    ax.set_xticklabels(np.arange(T))
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='salmon', label='Forward pass'),
        Patch(facecolor='lightblue', label='Backward pass'),
        Patch(facecolor='white', edgecolor='black', label='Bubble (idle)')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()


def compute_actual_bubble_time(timeline: np.ndarray) -> float:
    """Compute actual bubble fraction from timeline."""
    total_steps = timeline.size
    # timeline==0生成布尔矩阵,sum统计True的个数,即所有设备的空闲格子数
    bubble_steps = np.sum(timeline == 0)
    return bubble_steps / total_steps


# Visualize the pipeline we created earlier
print("Visualizing GPipe pipeline schedule...\n")

visualize_pipeline_schedule(pipeline_remat, f"GPipe: K={K} devices, M={M} micro-batches")

# Analyze bubble time
timeline = pipeline_remat.get_timeline_matrix()
actual_bubble = compute_actual_bubble_time(timeline)
theoretical_bubble = compute_bubble_fraction(K, M)

print(f"\nBubble Time Analysis (K={K}, M={M}):")
print(f"  Theoretical bubble fraction: {theoretical_bubble:.3f} ({theoretical_bubble*100:.1f}%)")
print(f"  Actual bubble fraction:      {actual_bubble:.3f} ({actual_bubble*100:.1f}%)")
print(f"  Pipeline efficiency:         {(1-actual_bubble)*100:.1f}%")

print("\n✓ Schedule visualization complete!")

# Section 7: Comparison - Pipeline vs Data Parallelism(第 7 节:对比——流水线并行 vs 数据并行)

Let's compare GPipe (pipeline parallelism) with traditional data parallelism.

下面我们将 GPipe(流水线并行)与传统的数据并行(data parallelism)进行对比。

## Data Parallelism(数据并行)
- Replicate entire model on each device
- Split batch across devices
- Synchronize gradients (all-reduce)
- **Limitation**: Model must fit on single device

- 在每个设备上复制完整模型
- 将 batch 切分到各个设备上
- 同步梯度(all-reduce)
- **局限**:模型必须能装进单个设备

## Pipeline Parallelism (GPipe)(流水线并行(GPipe))
- Split model across devices
- All devices work on same batch (different micro-batches)
- No gradient synchronization needed
- **Advantage**: Can train models larger than single device memory

- 将模型切分到多个设备上
- 所有设备处理同一个 batch(但处理不同的微批次)
- 不需要梯度同步
- **优势**:可以训练超过单设备内存容量的模型

#### 💻 代码解读

**做什么:** 用简化的计时模型对比两种并行方案——数据并行(每台设备存整个模型)和 GPipe 流水线并行(每台设备只存一段模型),量化各自的耗时与优劣。

**怎么做:**
- `simulate_data_parallelism` 假设每层耗时 1 个单位,总时间 = 前向 + 反向 + all-reduce 通信开销(各设备同步梯度),缺点是整个模型必须塞得进一台设备;
- `simulate_pipeline_parallelism` 按流水线公式算:前向/反向各需 `(K-1)+M` 步,每步耗时等于每分区层数,没有梯度同步的通信开销,但有气泡损耗;
- 用 4 台设备、批大小 32、8 个微批次跑两种模拟,分别打印耗时明细和效率;
- 结论:数据并行快但装不下大模型;流水线并行牺牲一点气泡时间,换来能训练 K 倍大的模型。

In [ ]:
def simulate_data_parallelism(model_layers: List[Layer], 
                             batch_size: int, 
                             num_devices: int) -> Dict[str, float]:
    """Simulate data parallelism timing.
    
    Returns:
        Dictionary with timing breakdown
    """
    # Each device processes batch_size/num_devices examples
    local_batch_size = batch_size // num_devices
    
    # Timing (arbitrary units)
    # 数据并行:每个设备跑完整模型(所有层),再用all-reduce同步梯度
    forward_time = len(model_layers) * 1.0  # One unit per layer
    backward_time = len(model_layers) * 1.0
    allreduce_time = 2.0  # Communication overhead
    
    total_time = forward_time + backward_time + allreduce_time
    
    return {
        'forward': forward_time,
        'backward': backward_time,
        'communication': allreduce_time,
        'total': total_time,
        'efficiency': (forward_time + backward_time) / total_time
    }


def simulate_pipeline_parallelism(model_layers: List[Layer],
                                 batch_size: int,
                                 num_devices: int,
                                 num_microbatches: int) -> Dict[str, float]:
    """Simulate pipeline parallelism timing."""
    layers_per_device = len(model_layers) // num_devices
    
    # Time for one micro-batch through one partition
    forward_time_per_micro = layers_per_device * 1.0
    backward_time_per_micro = layers_per_device * 1.0
    
    # Total pipeline time
    # Fill pipeline: (K-1) + M micro-batches
    # Each step: forward or backward through one partition
    # 流水线总步数=填充期(K-1)+稳定期M:先花K-1步让所有设备都有活干,之后每步流出一个micro-batch
    total_forward_steps = (num_devices - 1) + num_microbatches
    total_backward_steps = (num_devices - 1) + num_microbatches
    
    total_time = (total_forward_steps + total_backward_steps) * layers_per_device
    
    # Compute time (excluding bubbles)
    # 有效计算量=前向+反向(×2)×M个micro-batch×K个分区,用于算设备利用率
    compute_time = 2 * num_microbatches * layers_per_device * num_devices
    
    return {
        'forward': total_forward_steps * layers_per_device,
        'backward': total_backward_steps * layers_per_device,
        'communication': 0,  # No inter-device communication!
        'total': total_time,
        'efficiency': compute_time / (total_time * num_devices),
        'bubble_fraction': compute_bubble_fraction(num_devices, num_microbatches)
    }


# Compare both approaches
print("Comparing Pipeline Parallelism vs Data Parallelism\n")
print("="*70)

total_layers = 12
batch_size = 32
num_devices = 4
num_microbatches = 8

# Simulate data parallelism
data_parallel_stats = simulate_data_parallelism(model_layers, batch_size, num_devices)

print("Data Parallelism:")
print(f"  Configuration: {num_devices} devices, batch size {batch_size}")
print(f"  Forward time:        {data_parallel_stats['forward']:.1f} units")
print(f"  Backward time:       {data_parallel_stats['backward']:.1f} units")
print(f"  Communication time:  {data_parallel_stats['communication']:.1f} units (all-reduce)")
print(f"  Total time:          {data_parallel_stats['total']:.1f} units")
print(f"  Efficiency:          {data_parallel_stats['efficiency']*100:.1f}%")
print(f"  ⚠️  Limitation: Model must fit on single device!")

print("\n" + "="*70)

# Simulate pipeline parallelism
pipeline_stats = simulate_pipeline_parallelism(model_layers, batch_size, num_devices, num_microbatches)

print("Pipeline Parallelism (GPipe):")
print(f"  Configuration: {num_devices} devices, {num_microbatches} micro-batches")
print(f"  Forward time:        {pipeline_stats['forward']:.1f} units")
print(f"  Backward time:       {pipeline_stats['backward']:.1f} units")
print(f"  Communication time:  {pipeline_stats['communication']:.1f} units (none!)")
print(f"  Total time:          {pipeline_stats['total']:.1f} units")
print(f"  Efficiency:          {pipeline_stats['efficiency']*100:.1f}%")
print(f"  Bubble fraction:     {pipeline_stats['bubble_fraction']*100:.1f}%")
print(f"  ✓ Advantage: Can train models {num_devices}× larger!")

print("\n" + "="*70)
print("\nKey Differences:")
print("  • Data parallel: Fast, but model must fit on one device")
print("  • Pipeline parallel: Enables training of giant models")
print("  • GPipe: No communication overhead (unlike data parallel)")
print("  • Trade-off: Pipeline has bubble time, data parallel has communication")

print("\n✓ Comparison complete!")

# Section 8: Complete GPipe Training Loop(第 8 节:完整的 GPipe 训练循环)

Let's put it all together: a complete training loop with GPipe.

现在把所有部分组合起来:一个使用 GPipe 的完整训练循环。

#### 💻 代码解读

**做什么:** 把前面所有零件组装成完整的 GPipe 训练循环,在合成数据上真正训练 3 个轮次,验证损失确实在下降。

**怎么做:**
- `compute_loss` 对每个微批次算均方误差(MSE)再取平均,作为整个 mini-batch 的损失;
- `train_gpipe_epoch` 是一轮训练的完整流程:取 mini-batch → `split_into_microbatches` 切微批次 → `forward_pipeline_remat` 前向(带重计算)→ 算损失 → `backward_pipeline_remat` 反向 → `accumulate_gradients` 汇总梯度 → `apply_gradients` 更新参数;
- 生成 256 个随机样本的合成数据集,重新初始化 12 层模型并切成 K=4 个分区,配置批大小 32、微批次 8、学习率 0.001;
- 训练 3 个 epoch,每轮打印平均损失,观察损失逐轮下降。

In [ ]:
def compute_loss(outputs: List[np.ndarray], labels: List[np.ndarray]) -> float:
    """Compute average loss across micro-batches (MSE for simplicity)."""
    total_loss = 0.0
    # zip把输出列表和标签列表按micro-batch配对遍历
    for output, label in zip(outputs, labels):
        total_loss += np.mean((output - label) ** 2)
    return total_loss / len(outputs)


def train_gpipe_epoch(pipeline: GPipePipelineWithRemat,
                     X_train: np.ndarray,
                     y_train: np.ndarray,
                     batch_size: int,
                     num_microbatches: int,
                     learning_rate: float) -> List[float]:
    """Train one epoch with GPipe.
    
    Returns:
        List of losses for each mini-batch
    """
    num_samples = X_train.shape[0]
    num_batches = num_samples // batch_size
    
    losses = []
    
    # 完整的GPipe训练步骤:切micro-batch → 前向(存边界) → 反向(重计算) → 梯度累加 → 同步更新
    for batch_idx in range(num_batches):
        # Get mini-batch
        start = batch_idx * batch_size
        end = start + batch_size
        X_batch = X_train[start:end]
        y_batch = y_train[start:end]
        
        # Split into micro-batches
        microbatches = split_into_microbatches(X_batch, y_batch, num_microbatches)
        
        # Forward pass
        outputs, boundary_inputs = pipeline.forward_pipeline_remat(microbatches)
        
        # Compute loss
        # 列表推导式取出每个micro-batch元组的第2项(标签)
        labels = [mb[1] for mb in microbatches]
        loss = compute_loss(outputs, labels)
        losses.append(loss)
        
        # Backward pass
        all_gradients = pipeline.backward_pipeline_remat(outputs, labels, boundary_inputs)
        
        # Accumulate gradients
        accumulated_grads = accumulate_gradients(all_gradients)
        
        # Update parameters
        apply_gradients(pipeline.partitions, accumulated_grads, learning_rate)
    
    return losses


# Generate synthetic dataset
print("Creating synthetic dataset...\n")

num_train = 256
input_dim = 128
output_dim = 10

X_train = np.random.randn(num_train, input_dim)
y_train_labels = np.random.randint(0, output_dim, num_train)
# 花式索引生成one-hot标签矩阵,形状:(num_train,) -> (num_train, output_dim)
y_train = np.eye(output_dim)[y_train_labels]

print(f"Dataset: {num_train} samples, input dim {input_dim}, output dim {output_dim}")

# Create fresh model and pipeline
print("\nInitializing GPipe model...")

layer_dims = [input_dim] + [256] * 10 + [output_dim]
activations = ['relu'] * 10 + ['linear']
model_layers = create_model(layer_dims, activations)

K = 4
partitions = partition_model(model_layers, K)
pipeline = GPipePipelineWithRemat(partitions)

print(f"  Model: {len(model_layers)} layers")
print(f"  Partitions: {K} devices")

# Training configuration
batch_size = 32
num_microbatches = 8
learning_rate = 0.001
num_epochs = 3

print(f"\nTraining configuration:")
print(f"  Batch size: {batch_size}")
print(f"  Micro-batches: {num_microbatches}")
print(f"  Learning rate: {learning_rate}")
print(f"  Epochs: {num_epochs}")

# Train
print("\n" + "="*70)
print("Training GPipe model...")
print("="*70 + "\n")

all_losses = []

for epoch in range(num_epochs):
    pipeline.events = []  # Reset events for this epoch
    
    losses = train_gpipe_epoch(pipeline, X_train, y_train, 
                               batch_size, num_microbatches, learning_rate)
    
    avg_loss = np.mean(losses)
    all_losses.extend(losses)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Average Loss = {avg_loss:.6f}")

print("\n✓ Training complete!")

# Section 9: Visualizations and Analysis(第 9 节:可视化与分析)

Let's create comprehensive visualizations of GPipe's performance.

下面我们对 GPipe 的性能进行全面的可视化分析。

#### 💻 代码解读

**做什么:** 画一张 2×2 的总结图,从四个角度可视化 GPipe 的训练效果和核心权衡。

**怎么做:**
- 左上:用 `all_losses` 画训练损失曲线,确认模型在学习;
- 右上:对 K=2/4/8/16 各画一条曲线,展示气泡占比随微批次数 M(1~64)的变化——M 越大气泡越小,K 越大气泡越大;
- 左下:用 `estimate_memory_usage` 对比"开/不开重计算"时内存随分区数 K 的增长,开重计算的那条线明显更省内存;
- 右下:画流水线效率(1 − 气泡率)随 K 的变化,每个 M 值一条线,说明"设备越多,越需要更多微批次来喂饱流水线"。

In [ ]:
# Visualization 1: Training Loss Curve
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Training loss
ax = axes[0, 0]
ax.plot(all_losses, linewidth=2, color='darkblue')
ax.set_xlabel('Mini-batch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('GPipe Training Loss', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: Bubble fraction vs M (micro-batches)
ax = axes[0, 1]
M_range = np.arange(1, 65)
K_values_plot = [2, 4, 8, 16]
colors = ['blue', 'green', 'orange', 'red']

# zip同时遍历K值和对应颜色;每条曲线展示固定K下气泡率随M增大而下降
for K_val, color in zip(K_values_plot, colors):
    bubbles = [compute_bubble_fraction(K_val, M) for M in M_range]
    ax.plot(M_range, bubbles, label=f'K={K_val}', linewidth=2, color=color)

ax.set_xlabel('Number of Micro-batches (M)', fontsize=11)
ax.set_ylabel('Bubble Fraction', fontsize=11)
ax.set_title('Bubble Time vs Micro-batches', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

# Plot 3: Memory savings with re-materialization
ax = axes[1, 0]
K_range = np.arange(2, 17)
layers_per_partition = 3
M_fixed = 8
activation_size_mb = 10

# 列表推导式对每个K值算一次显存估计,对比是否使用重计算
mem_without_remat = [estimate_memory_usage(M_fixed, K_val, layers_per_partition, 
                                            activation_size_mb, False) 
                     for K_val in K_range]
mem_with_remat = [estimate_memory_usage(M_fixed, K_val, layers_per_partition, 
                                        activation_size_mb, True) 
                  for K_val in K_range]

ax.plot(K_range, mem_without_remat, label='Without Remat', linewidth=2, 
        marker='o', color='red', markersize=6)
ax.plot(K_range, mem_with_remat, label='With Remat', linewidth=2, 
        marker='s', color='green', markersize=6)
ax.set_xlabel('Number of Partitions (K)', fontsize=11)
ax.set_ylabel('Memory (MB)', fontsize=11)
ax.set_title('Memory Usage: Re-materialization Impact', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Pipeline efficiency vs configuration
ax = axes[1, 1]
M_configs = [4, 8, 16, 32]
K_configs = np.arange(2, 17)

for M_val in M_configs:
    # 效率=1-气泡率;可以看到K越大效率越低,需要更大的M补偿
    efficiencies = [1 - compute_bubble_fraction(K_val, M_val) for K_val in K_configs]
    ax.plot(K_configs, efficiencies, label=f'M={M_val}', linewidth=2, marker='o', markersize=5)

ax.set_xlabel('Number of Devices (K)', fontsize=11)
ax.set_ylabel('Pipeline Efficiency', fontsize=11)
ax.set_title('Pipeline Efficiency vs Configuration', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\n✓ Visualizations complete!")

# Section 10: Key Insights and Modern Extensions(第 10 节:关键洞见与现代扩展)

## Summary of GPipe(GPipe 总结)

### Core Ideas(核心思想)
1. **Pipeline Parallelism**: Split model across devices by layers
2. **Micro-batching**: Split mini-batches to reduce bubble time
3. **Re-materialization**: Trade computation for memory efficiency
4. **F-then-B Schedule**: Forward all micro-batches, then backward all

1. **流水线并行(pipeline parallelism)**:按层将模型切分到多个设备上
2. **微批次(micro-batching)**:切分 mini-batch 以减少气泡时间
3. **重计算(re-materialization)**:用计算换取内存效率
4. **F-then-B 调度**:先对所有微批次做前向,再对所有微批次做反向

### Mathematical Insights(数学洞见)

**Bubble Fraction**:
$$\text{Bubble} = \frac{K-1}{K-1+M}$$

**Memory Savings** (with re-materialization):
$$\text{Memory}_{\text{remat}} = \frac{K}{L} \times \text{Memory}_{\text{standard}}$$

where L = total layers, K = partitions.

**Speedup** (compared to single device):
$$\text{Speedup} \approx \frac{K}{1 + \frac{K-1}{M}}$$

**气泡占比(bubble fraction)**如第一个公式所示;**内存节省**(使用重计算时)如第二个公式所示,其中 L 为总层数,K 为分区数;**加速比(speedup)**(相对于单设备)如第三个公式所示。

### When to Use GPipe(何时使用 GPipe)

**Use GPipe when**:
- Model doesn't fit on single device
- Sequential model structure (layers)
- Limited inter-device bandwidth
- Can use large M (many micro-batches)

**Avoid GPipe when**:
- Model fits on single device (use data parallel instead)
- Very small M (bubble time dominates)
- Non-sequential architecture (e.g., heavy skip connections)

**适合使用 GPipe 的情况**:
- 模型装不进单个设备
- 模型结构是顺序式的(逐层)
- 设备间带宽有限
- 可以使用较大的 M(很多微批次)

**不适合使用 GPipe 的情况**:
- 模型能装进单个设备(此时应使用数据并行)
- M 非常小(气泡时间占主导)
- 非顺序式架构(例如大量跳跃连接 skip connections)

---

## Modern Extensions(现代扩展)

### 1. PipeDream (Harlap et al., 2018)
- **1F1B schedule**: Interleave forward and backward
- Reduces pipeline depth
- Better memory efficiency

- **1F1B 调度**:交替执行前向和反向
- 减小流水线深度
- 更好的内存效率

### 2. Megatron-LM (Shoeybi et al., 2019)
- Combines pipeline + tensor parallelism
- Splits layers horizontally (within layer)
- Used for 530B parameter models

- 结合流水线并行与张量并行(tensor parallelism)
- 对层进行水平切分(层内切分)
- 曾用于训练 5300 亿参数的模型

### 3. ZeRO (Rajbhandari et al., 2020)
- Partitions optimizer states, gradients, parameters
- Complements pipeline parallelism
- Reduces memory without replication

- 对优化器状态、梯度、参数进行分区
- 与流水线并行互补
- 无需复制即可降低内存占用

### 4. Varuna (Athlur et al., 2022)
- Automatic pipeline schedule optimization
- Adaptive micro-batching
- Handles heterogeneous devices

- 自动优化流水线调度
- 自适应微批次
- 支持异构设备

---

## Practical Considerations(实践考量)

### Optimal M (micro-batches)(最优的 M(微批次数))
- **Too small**: High bubble fraction
- **Too large**: Overhead from micro-batch management
- **Rule of thumb**: M ≈ 4×K

- **太小**:气泡占比高
- **太大**:微批次管理带来额外开销
- **经验法则**:M ≈ 4×K

### Partitioning Strategy(切分策略)
- Uniform: Equal layers per device
- Balanced: Equal computation time per device
- Memory-aware: Balance memory usage

- 均匀切分:每个设备层数相同
- 均衡切分:每个设备计算时间相同
- 内存感知切分:均衡内存使用

### Batch Size(批大小)
- Large batches improve pipeline utilization
- But may hurt generalization
- Compensate with learning rate scaling

- 大 batch 可提高流水线利用率
- 但可能损害泛化能力
- 可通过学习率缩放(learning rate scaling)来补偿

---

## Connection to Other Papers(与其他论文的联系)

**Paper 5 (Optimal Brain Damage)**: Pruning reduces model size → less pipeline stages needed

**Paper 23 (MDL)**: Model complexity vs data fit → choosing K (partitions) involves trade-off

**Paper 14 (Neural Architecture Search)**: Can use GPipe to search architectures too large for single device

**论文 5(Optimal Brain Damage)**:剪枝(pruning)减小模型规模 → 需要的流水线阶段更少

**论文 23(MDL)**:模型复杂度与数据拟合的权衡 → 选择 K(分区数)同样涉及权衡

**论文 14(神经架构搜索,Neural Architecture Search)**:可以用 GPipe 搜索单设备装不下的大型架构

---

## Real-World Impact(现实影响)

GPipe enabled:
- **AmoebaNet-B**: 557M parameters (8× larger than previous best)
- **Trained on ImageNet** with 84.4% top-1 accuracy
- **GPT-3**: 175B parameters (combination of techniques including pipeline parallelism)
- **Large language models**: Modern LLMs use pipeline + tensor + data parallelism

GPipe 促成了:
- **AmoebaNet-B**:5.57 亿参数(比此前最佳模型大 8 倍)
- **在 ImageNet 上训练**,达到 84.4% 的 top-1 准确率
- **GPT-3**:1750 亿参数(结合了包括流水线并行在内的多种技术)
- **大语言模型**:现代 LLM 综合使用流水线并行 + 张量并行 + 数据并行

---

**GPipe's Legacy**: Showed that **model parallelism is practical** and paved the way for training models with hundreds of billions of parameters. Combined with tensor parallelism and ZeRO, it forms the foundation of modern large-scale training!

**GPipe 的遗产**:它证明了**模型并行(model parallelism)是切实可行的**,为训练数千亿参数的模型铺平了道路。与张量并行和 ZeRO 相结合,它构成了现代大规模训练的基石!

#### 💻 代码解读

**做什么:** 收尾总结:打印一份 GPipe 配置指南,教你实际使用时如何选择设备数 K 和微批次数 M。

**怎么做:**
- 先列出选 K 的考虑:K 受手头加速器数量限制,K 越大能训练越大的模型,但气泡也越多;
- 再给出选 M 的经验法则:M ≈ 4×K——M 越大气泡越少但开销越多,且 M 必须能整除批大小;
- 用 `compute_bubble_fraction` 对四组典型配置(K=2/4/8/16 搭配 M=8/16/32/64)算出效率和气泡率,打印成对照表,验证 4×K 法则下效率都稳定在约 88.9%。

In [ ]:
# Final demonstration: Show trade-off between K and M
print("="*70)
print("GPipe Configuration Guide")
print("="*70)

print("\n1. Choosing K (number of devices):")
print("   • Limited by: Number of available accelerators")
print("   • More K = Can train larger models")
print("   • More K = More bubble time (need larger M to compensate)")

print("\n2. Choosing M (number of micro-batches):")
print("   • Rule of thumb: M ≈ 4×K")
print("   • Larger M = Less bubble time")
print("   • Larger M = More overhead")
print("   • Must divide batch size evenly")

print("\n3. Example configurations:")
configs = [
    (2, 8, 32),
    (4, 16, 64),
    (8, 32, 128),
    (16, 64, 256),
]

# 元组解包:每个配置(K, M, batch)直接拆成三个变量;这些配置遵循经验法则M≈4K
for K, M, batch in configs:
    bubble = compute_bubble_fraction(K, M)
    efficiency = 1 - bubble
    print(f"   K={K:2d}, M={M:2d}, batch={batch:3d} → "
          f"Efficiency={efficiency*100:.1f}%, Bubble={bubble*100:.1f}%")

print("\n" + "="*70)
print("✓ GPipe implementation complete!")
print("="*70)